In [ ]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.getOrCreate()
from pyspark.sql.functions import col, xxhash64
from helper_utils.functions import get_incremental_data, deduplicate, PIPELINE_START_DATE, return_pipeline_upper_bound
from delta.tables import DeltaTable
from yellow_taxi.yellow_taxi_utils import normalize_yellow_trips, merge_insert

In [ ]:
dbutils.widgets.text("run_date", "2026-10-07") # default end date for testing
run_date = dbutils.widgets.get("run_date")

# if run in october, the lag (3 months) should resolve to july (in october we accept all the records from july and date must be 2026-08-01)

In [ ]:
# variables

min_pickup_ts = PIPELINE_START_DATE
max_pickup_ts = return_pipeline_upper_bound(run_date)

# interactive widgets for catalog and schema names

dbutils.widgets.text("catalog", "nyctaxi_project_dev")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")

# dynamic variables for catalog and schema names
catalog = dbutils.widgets.get("catalog")
source_schema = dbutils.widgets.get("bronze_schema")
target_schema = dbutils.widgets.get("silver_schema")

source_table = f"{catalog}.{source_schema}.yellow_trips_raw"
target_table = f"{catalog}.{target_schema}.yellow_trips_normalized"


In [4]:
# load bronze trips, cast data types, filter for expected time range
bronze_df = spark.table(source_table)

# identify only incremental trips or load the whole dataset if it's the first load
incremental_trips = get_incremental_data(spark, target_table, bronze_df)

# cast data types and filter for valid trips
normalized_df = normalize_yellow_trips(incremental_trips, min_pickup_ts, max_pickup_ts)
# display(normalized_df)

In [5]:
fillna_dict = {
'vendor_id': 99,
 'rate_code_id': 99,
 'pickup_location_id': 264,
 'dropoff_location_id': 264,
 'payment_type': 5,
 'extra': 0,
 'mta_tax': 0,
 'tip_amount': 0,
 'tolls_amount': 0,
 'improvement_surcharge': 0,
 'congestion_surcharge': 0,
 'airport_fee': 0,
 'cbd_congestion_fee': 0
}

In [6]:
# fill in missing values to avoid unexpected result 

df_fill_na = normalized_df.fillna(value=fillna_dict)

In [7]:
# create a unique identifier for each trip using xxhash64
id_col= "trip_id"
hash_on_cols = [
            col("vendor_id"),
            col("pickup_datetime"),
            col("dropoff_datetime"),
            col("passenger_count"),
            col("trip_distance"),
            col("rate_code_id"),
            col("pickup_location_id"),
            col("dropoff_location_id"),
            col("payment_type"),
            col("fare_amount"),
            col("extra"),
            col("mta_tax"),
            col("tip_amount"),
            col("tolls_amount"),
            col("improvement_surcharge"),
            col("total_amount"),
            col("congestion_surcharge"),
            col("airport_fee"),
            col("cbd_congestion_fee")
            ]

df_with_trip_id = df_fill_na.withColumn(id_col, xxhash64(*hash_on_cols))

# display(df_with_trip_id)

In [8]:
# remove duplicated

partition_by_col= "trip_id"
order_by_col = "_ingestion_timestamp"

df_deduplicated = deduplicate(df_with_trip_id, partition_by_col, order_by_col)
                       
# display(df_deduplicated)

In [9]:
df_final = (
    df_deduplicated 
        .select(
            "trip_id",
            "vendor_id",
            "pickup_datetime",
            "dropoff_datetime",
            "passenger_count",
            "trip_distance",
            "rate_code_id",
            "pickup_location_id",
            "dropoff_location_id",
            "payment_type",
            "fare_amount",
            "extra",
            "mta_tax",
            "tip_amount",
            "tolls_amount",
            "improvement_surcharge",
            "total_amount",
            "congestion_surcharge",
            "airport_fee",
            "cbd_congestion_fee",
            "_ingestion_timestamp",
            "_source_file_name",
            "_source_file_path",
            "_source_file_last_modified"          
        )
)

In [11]:
# if table exists, merge the new data with the existing data, otherwise create a new table

if not spark.catalog.tableExists(target_table):
    df_final.write.mode("overwrite").saveAsTable(target_table)
else:
    dt = DeltaTable.forName(spark, target_table)
    merge_insert(df_final, dt)